# Demonstrate how to convert healpix to lat/lon and then to iris cube

* Computes number of latitude and longitude points equating to input zoom (or define an analysis grid)
* Uses nearest neighbour interpolation.
* Plots the results using iris plotting.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import intake
import matplotlib.pyplot as plt
import numpy as np

import healpy as hp
import xarray as xr
import iris
import iris.plot as iplt
import iris.quickplot as qplt

from utils import hp_mods, plot_all_fields

In [ ]:
# Filter out annoying warning.
import warnings
warnings.filterwarnings("ignore", message=".*The return type of `Dataset.dims` will be changed.*", category=FutureWarning)

In [ ]:
# Define function to compute number of latitude and longitude points appropriate for each Healpix zoom
# For more approximate computation, nlon ~ 4*nside; nlat ~ 3*nside
def healpix_zoom_to_grid_area_match(zoom):
    nside = 2.0**zoom

    # Healpix pixel area
    pixel_area = 4 * np.pi / (12 * nside**2)

    # Angular resolution
    theta = np.sqrt(pixel_area)

    # Compute nlon, nlat
    nlon = int(round(2 * np.pi / theta))
    nlat = int(round(np.pi / theta))

    return nlon, nlat

In [ ]:
# Define function to translate from healpix to lat lon [could sit in utils.py, but showing here for transparency]
def get_nn_lon_lat_index(nside, lons, lats):
    lons2, lats2 = np.meshgrid(lons, lats)
    return xr.DataArray(
        hp.ang2pix(nside, lons2, lats2, nest=True, lonlat=True),
        coords=[("latitude", lats), ("longitude", lons)],
    )

In [ ]:
# Define a function to generate iris cube from xarray Dataset
def xarray_to_iris(dataset, varname):

    # Clean up healpix attributes
    dataset["cell"].attrs.pop("standard_name", None)
    dataset["lat"].attrs.pop("standard_name", None)
    dataset["lon"].attrs.pop("standard_name", None)
    
    # Convert from xarray to iris cube
    cube = dataset[varname].to_iris()

    # Add more metadata to lat and lon coords
    for cube_coord in ["latitude", "longitude"]:
        cube.coord(cube_coord).units="degrees"
        cube.coord(cube_coord).coord_system=iris.coord_systems.GeogCS(6371229.0)
        cube.coord(cube_coord).guess_bounds()
        
    # Remove crs coord
    cube.remove_coord("crs")
    
    # Set standard_calendar for time coord
    cube.coord("time").units = cube.coord("time").units.change_calendar("standard")

    return cube

In [ ]:
# Open catalog.
url = 'https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml'
cat = intake.open_catalog(url)['UK']
# Use online if not on JASMIN.
# cat = intake.open_catalog('https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml')['online']

In [ ]:
# Print all catalog entries
print([k for k in cat])

In [ ]:
# Print all hk26 catalog entries
print('\n'.join([k for k in cat if k.endswith('_hk26')]))

In [ ]:
# Load specific model
sim = 'um_glm_n2560_RAL3p3_tuned_hk26'
sim_cat = cat()[sim]

In [ ]:
# Open a 1h (2D) and 3h (3D) dataset - work with lo-res and hi-res example healpix
zoom_lr = 2
ds1h_lr = sim_cat(zoom=zoom_lr, time='PT1H').to_dask().pipe(hp_mods)
ds3h_lr = sim_cat(zoom=zoom_lr, time='PT3H').to_dask().pipe(hp_mods)

In [ ]:
# Open a 1h (2D) and 3h (3D) dataset - work with lo-res and hi-res example healpix
zoom_hr = 8
ds1h_hr = sim_cat(zoom=zoom_hr, time='PT1H').to_dask().pipe(hp_mods)
ds3h_hr = sim_cat(zoom=zoom_hr, time='PT3H').to_dask().pipe(hp_mods)

In [ ]:
ds1h_hr

In [ ]:
# OPTION A: Define an analyisis grid (e.g. 1 degree global) - used to output all zoom levels to same analysis grid.
#           Caution uses nearest neighbour interpolation. May be useful for exploring data and testing ideas, or 
#           where matching to e.g. observations grid.
#           Either set bounds [lon1, lon2, lat1, lat2] based on input model domain extent, or user-specified
import ast
domain_bounds = ast.literal_eval(ds1h_hr.attrs.get("regional_bounds"))
lon1 = domain_bounds["lower_left_lon"]
lon2 = domain_bounds["upper_right_lon"]
nlon = 360
lons = np.linspace(lon1, lon2, nlon)

lat1 = domain_bounds["lower_left_lat"]
lat2 = domain_bounds["upper_right_lat"]
nlat = 180
lats = np.linspace(lat1, lat2, nlat)

# Get healpix_index coord corresponding to lat/lon mesh
idx_lr = get_nn_lon_lat_index(2**zoom_lr, lons, lats)
idx_hr = get_nn_lon_lat_index(2**zoom_hr, lons, lats)

In [ ]:
len(idx_lr)

In [ ]:
len(idx_hr)

In [ ]:
ds1h_lr_latlon = ds1h_lr.isel(cell=idx_lr)
ds1h_hr_latlon = ds1h_hr.isel(cell=idx_hr)

ds3h_lr_latlon = ds3h_lr.isel(cell=idx_lr)
ds3h_hr_latlon = ds3h_hr.isel(cell=idx_hr)

In [ ]:
ds1h_lr_latlon

In [ ]:
#ds1h_lr_latlon["cell"].attrs.pop("standard_name", None)
#temp_cube_latlon_lr_A = ds1h_lr_latlon["tas"].to_iris()

In [ ]:
temp_cube_latlon_lr_A = xarray_to_iris(ds1h_lr_latlon, "tas")

In [ ]:
#ds1h_hr_latlon["cell"].attrs.pop("standard_name", None)
#temp_cube_latlon_hr_A = ds1h_hr_latlon["tas"].to_iris()

In [ ]:
temp_cube_latlon_hr_A = xarray_to_iris(ds1h_hr_latlon, "tas")

In [ ]:
temp_cube_latlon_lr_A

In [ ]:
temp_cube_latlon_hr_A

In [ ]:
# Use iris plotting function with some user-defined options
# See https://scitools-iris.readthedocs.io/en/stable/ for more information on iris
plt.figure(figsize=(8,6))
iplt.pcolormesh(temp_cube_latlon_lr_A[1], vmin=210, vmax=310, cmap='RdYlBu_r')
plt.colorbar(orientation='horizontal')

In [ ]:
# Use iris plotting function with some user-defined options
# See https://scitools-iris.readthedocs.io/en/stable/ for more information on iris
plt.figure(figsize=(8,6))
iplt.pcolormesh(temp_cube_latlon_hr_A[1], vmin=210, vmax=310, cmap='RdYlBu_r')
plt.colorbar(orientation='horizontal')

In [ ]:
# More user-specified iris plotting options using cartopy
# See https://cartopy.readthedocs.io/stable/
fig = plt.figure(figsize=(10,5))
projection = ccrs.PlateCarree(central_longitude=0)
crs = ccrs.PlateCarree()
axes = fig.add_subplot(projection=projection)
iplt.pcolormesh(temp_cube_latlon_hr_A[1], vmin=210, vmax=310, cmap='RdYlBu_r')
axes.coastlines(resolution="10m", color='k')
axes.add_feature(cfeature.BORDERS, edgecolor='k')
plt.colorbar(orientation='horizontal',shrink=0.7, pad=0.042)

In [ ]:
# More user-specified iris plotting options using cartopy
# See https://cartopy.readthedocs.io/stable/
fig = plt.figure(figsize=(10,5))
projection = ccrs.PlateCarree(central_longitude=0)
crs = ccrs.PlateCarree()
axes = fig.add_subplot(projection=projection)
iplt.pcolormesh(temp_cube_latlon_lr_A[1], vmin=210, vmax=310, cmap='RdYlBu_r')
axes.coastlines(resolution="10m", color='k')
axes.add_feature(cfeature.BORDERS, edgecolor='k')
plt.colorbar(orientation='horizontal',shrink=0.7, pad=0.042)

In [ ]:
# OPTION B: (Recommended) Compute number of lat and lon points equivalent to Healpix zoom level
#          Note option to define lons [0,360] or [-180,180]
#          Either set bounds [lon1, lon2, lat1, lat2] based on input model domain extent, or user-specified
domain_bounds = ast.literal_eval(ds1h_hr.attrs.get("regional_bounds"))
lon1 = domain_bounds["lower_left_lon"]
lon2 = domain_bounds["upper_right_lon"]
lat1 = domain_bounds["lower_left_lat"]
lat2 = domain_bounds["upper_right_lat"]

# Call function to compute nlon, nlat based on zoom level
nlon_lr, nlat_lr = healpix_zoom_to_grid_area_match(zoom_lr) #, lat1, lat2, lon1, lon2)
lons_lr = np.linspace(lon1, lon2, nlon_lr)
lats_lr = np.linspace(lat1, lat2, nlat_lr)

nlon_hr, nlat_hr = healpix_zoom_to_grid_area_match(zoom_hr) #, lat1, lat2, lon1, lon2)
lons_hr = np.linspace(lon1, lon2, nlon_hr)
lats_hr = np.linspace(lat1, lat2, nlat_hr)

# Get healpix_index coord corresponding to lat/lon mesh
idx_lr = get_nn_lon_lat_index(2**zoom_lr, lons_lr, lats_lr)
idx_hr = get_nn_lon_lat_index(2**zoom_hr, lons_hr, lats_hr)

In [ ]:
len(idx_lr)

In [ ]:
len(idx_hr)

In [ ]:
ds1h_lr_latlon = ds1h_lr.sel(cell=idx_lr)
ds3h_lr_latlon = ds3h_lr.sel(cell=idx_lr)

In [ ]:
ds1h_hr_latlon = ds1h_hr.sel(cell=idx_hr)
ds3h_hr_latlon = ds3h_hr.sel(cell=idx_hr)

In [ ]:
temp_cube_latlon_lr_B = xarray_to_iris(ds1h_lr_latlon, "tas")

In [ ]:
temp_cube_latlon_hr_B = xarray_to_iris(ds1h_hr_latlon, "tas")

In [ ]:
# Use iris plotting function with some user-defined options
# See https://scitools-iris.readthedocs.io/en/stable/ for more information on iris
plt.figure(figsize=(8,6))
iplt.pcolormesh(temp_cube_latlon_lr_B[1], vmin=210, vmax=310, cmap='RdYlBu_r')
plt.colorbar(orientation='horizontal')

In [ ]:
# Use iris plotting function with some user-defined options
# See https://scitools-iris.readthedocs.io/en/stable/ for more information on iris
plt.figure(figsize=(8,6))
iplt.pcolormesh(temp_cube_latlon_hr_B[1], vmin=210, vmax=310, cmap='RdYlBu_r')
plt.colorbar(orientation='horizontal')

In [ ]:
# More user-specified iris plotting options using cartopy
# See https://cartopy.readthedocs.io/stable/
fig = plt.figure(figsize=(10,5))
projection = ccrs.PlateCarree(central_longitude=0)
crs = ccrs.PlateCarree()
axes = fig.add_subplot(projection=projection)
iplt.pcolormesh(temp_cube_latlon_hr_B[1], vmin=210, vmax=310, cmap='RdYlBu_r')
axes.coastlines(resolution="10m", color='k')
axes.add_feature(cfeature.BORDERS, edgecolor='k')
plt.colorbar(orientation='horizontal',shrink=0.7, pad=0.042)

In [ ]:
# Use iris analysis to compute global average timeseries (only compute for lo-res input given global mean output)
t_mean_lr_A = temp_cube_latlon_lr_A.collapsed(["latitude","longitude"], iris.analysis.MEAN)
t_mean_lr_B = temp_cube_latlon_lr_B.collapsed(["latitude","longitude"], iris.analysis.MEAN)

In [ ]:
# Use iris analysis to compute global average timeseries (try hi-res input given global mean output)
t_mean_hr_A = temp_cube_latlon_hr_A.collapsed(["latitude","longitude"], iris.analysis.MEAN)
t_mean_hr_B = temp_cube_latlon_hr_B.collapsed(["latitude","longitude"], iris.analysis.MEAN)

In [ ]:
fig = plt.figure(figsize=(10,5))
iplt.plot(t_mean_lr_A, label="zoom 4, 360 x 180 grid")
iplt.plot(t_mean_lr_B, label="zoom 4, nlon x nlat grid")
iplt.plot(t_mean_hr_A, label="zoom 8, 360 x 180 grid")
iplt.plot(t_mean_hr_B, label="zoom 8, nlon x nlat grid")
plt.grid()
plt.legend()